# Assignment 2: Convolutional Neural Networks

**Deep Learning FS26**

---

## Part 1: Task Description

### Problem Description

In this assignment you will implement and train Convolutional Neural Networks (CNNs) for image classification. You will build a CNN from scratch using PyTorch, apply it to the CIFAR-10 dataset, and study the effect of different architectural choices.

### Tasks Overview

1. **Manual Convolution**
   - Implement a 2D convolution operation manually using NumPy (without `torch.nn.Conv2d`).
   - Apply common filters (edge detection, blur) to a sample image.
   - Visualize the filtered outputs.

2. **CNN Architecture in PyTorch**
   - Build a CNN with the following structure:
     - Conv layer → ReLU → MaxPool → Conv layer → ReLU → MaxPool → Flatten → FC → ReLU → FC → Softmax
   - Train on the **CIFAR-10** dataset.
   - Use `CrossEntropyLoss` and the `Adam` optimizer.

3. **Feature Map Visualization**
   - Visualize the feature maps (activation outputs) produced by the first convolutional layer.
   - Interpret what patterns different filters detect.

### Possible Solutions

- The manual convolution should produce the same result as `torch.nn.functional.conv2d` for verification.
- Your CNN should reach at least **60% test accuracy** on CIFAR-10 after 10 epochs.
- Feature maps in early layers should highlight edges, colors, and textures.

### Expected Plots

- **Filter response images**: Side-by-side comparison of the original image and the filtered output (edge detection, Gaussian blur).

  ```
  [Original Image]   [Edge Filter]   [Blur Filter]
  ```

- **Training & Validation Curves**: Loss and accuracy per epoch for both training and validation sets.

  ```
  Accuracy
  1.0 |          ___train
      |      ___/
  0.6 |  ___/ ___val
      | / ___/
  0.0 |/________
       0   5   10  Epoch
  ```

- **Feature map grid**: A grid of 16–32 feature maps from the first conv layer for a single input image.

## Part 2: Implementation

### Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
np.random.seed(42)

### Task 1: Manual 2D Convolution

In [ ]:
def conv2d_manual(image, kernel, stride=1, padding=0):
    """
    Perform a 2D convolution on a 2D image with the given kernel.
    
    Args:
        image: 2D numpy array of shape (H, W)
        kernel: 2D numpy array of shape (kH, kW)
        stride: convolution stride (default 1)
        padding: zero-padding size (default 0)
    
    Returns:
        output: 2D numpy array with the convolution result
    """
    # TODO: Implement 2D convolution
    H, W = image.shape
    kH, kW = kernel.shape

    if padding > 0:
        image = np.pad(image, padding, mode='constant')

    out_H = (H + 2 * padding - kH) // stride + 1
    out_W = (W + 2 * padding - kW) // stride + 1
    output = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            region = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            output[i, j] = np.sum(region * kernel)

    return output


# Define standard filters
edge_kernel = np.array([[-1, -1, -1],
                        [-1,  8, -1],
                        [-1, -1, -1]])

blur_kernel = np.ones((5, 5)) / 25.0

# Load a sample image using torchvision
sample_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True,
    transform=transforms.ToTensor()
)
sample_img, sample_label = sample_dataset[0]
# Convert to grayscale for convolution demo
sample_gray = sample_img.mean(dim=0).numpy()  # (32, 32)

# Apply filters
edge_output = conv2d_manual(sample_gray, edge_kernel, padding=1)
blur_output = conv2d_manual(sample_gray, blur_kernel, padding=2)

# Visualize
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_gray, cmap='gray')
axes[0].set_title(f'Original ({classes[sample_label]})')
axes[1].imshow(edge_output, cmap='gray')
axes[1].set_title('Edge Detection Filter')
axes[2].imshow(blur_output, cmap='gray')
axes[2].set_title('Gaussian Blur Filter')
for ax in axes:
    ax.axis('off')
plt.suptitle('Manual 2D Convolution Results', fontsize=14)
plt.tight_layout()
plt.show()

### Task 2: CNN on CIFAR-10

In [ ]:
# Data loading with augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader  = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # TODO: Define convolutional and fully connected layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # TODO: Implement forward pass
        x = self.pool(F.relu(self.conv1(x)))  # (B, 32, 16, 16)
        x = self.pool(F.relu(self.conv2(x)))  # (B, 64, 8, 8)
        x = x.view(x.size(0), -1)            # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total += inputs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
            total += inputs.size(0)
    return total_loss / total, correct / total


# Train the model
cnn = SimpleCNN().to(device)
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

num_epochs = 10
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, num_epochs + 1):
    tr_loss, tr_acc = train_epoch(cnn, trainloader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(cnn, testloader, criterion, device)
    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)
    print(f'Epoch {epoch:02d} | '
          f'Train Loss: {tr_loss:.3f}, Train Acc: {tr_acc:.3f} | '
          f'Val Loss: {va_loss:.3f}, Val Acc: {va_acc:.3f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, num_epochs+1), train_losses, label='Train Loss', color='steelblue')
ax1.plot(range(1, num_epochs+1), val_losses, label='Val Loss', color='coral', linestyle='--')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Loss Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, num_epochs+1), [a*100 for a in train_accs], label='Train Acc', color='steelblue')
ax2.plot(range(1, num_epochs+1), [a*100 for a in val_accs], label='Val Acc', color='coral', linestyle='--')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('CNN Training on CIFAR-10', fontsize=14)
plt.tight_layout()
plt.show()

### Task 3: Feature Map Visualization

In [ ]:
# Extract feature maps from the first conv layer
cnn.eval()

# Pick a test image
test_img_tensor, test_label = testset[0]
test_img_tensor = test_img_tensor.unsqueeze(0).to(device)

# Hook to capture intermediate activations
feature_maps = {}

def get_activation(name):
    def hook(model, input, output):
        feature_maps[name] = output.detach()
    return hook

hook_handle = cnn.conv1.register_forward_hook(get_activation('conv1'))

with torch.no_grad():
    _ = cnn(test_img_tensor)

hook_handle.remove()

# Visualize feature maps
fmaps = feature_maps['conv1'].squeeze(0).cpu().numpy()  # (32, H, W)
n_maps = min(16, fmaps.shape[0])

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i < n_maps:
        ax.imshow(fmaps[i], cmap='viridis')
        ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Conv1 Feature Maps — "{classes[test_label]}"', fontsize=14)
plt.tight_layout()
plt.show()

## Part 3: Experiments and Analysis

### Experiment 1: Effect of Batch Normalization

Add Batch Normalization (`nn.BatchNorm2d`) after each convolutional layer and compare training speed and final accuracy against the baseline CNN.

In [ ]:
class CNNWithBatchNorm(nn.Module):
    def __init__(self):
        super(CNNWithBatchNorm, self).__init__()
        # TODO: Add BatchNorm layers after each conv layer
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64 * 8 * 8, 256)
        self.fc2   = nn.Linear(256, 10)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


cnn_bn = CNNWithBatchNorm().to(device)
optimizer_bn = optim.Adam(cnn_bn.parameters(), lr=1e-3)

bn_train_losses, bn_val_losses = [], []
bn_train_accs, bn_val_accs = [], []

for epoch in range(1, num_epochs + 1):
    tr_loss, tr_acc = train_epoch(cnn_bn, trainloader, optimizer_bn, criterion, device)
    va_loss, va_acc = evaluate(cnn_bn, testloader, criterion, device)
    bn_train_losses.append(tr_loss)
    bn_val_losses.append(va_loss)
    bn_train_accs.append(tr_acc)
    bn_val_accs.append(va_acc)
    print(f'[BN] Epoch {epoch:02d} | Train Acc: {tr_acc:.3f} | Val Acc: {va_acc:.3f}')

In [ ]:
# Compare baseline CNN vs CNN with BatchNorm
plt.figure(figsize=(10, 4))
plt.plot([a*100 for a in val_accs], label='Baseline CNN', color='steelblue', linewidth=2)
plt.plot([a*100 for a in bn_val_accs], label='CNN + BatchNorm', color='coral', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy (%)')
plt.title('Validation Accuracy: Baseline vs Batch Normalization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# TODO: Describe what you observe
# YOUR ANSWER HERE:

### Experiment 2: Confusion Matrix

Compute and plot the confusion matrix for the trained CNN on the test set.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TODO: Collect all predictions on the test set
all_preds, all_targets = [], []
cnn.eval()
with torch.no_grad():
    for inputs, targets in testloader:
        inputs = inputs.to(device)
        outputs = cnn(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.numpy())

cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Confusion Matrix — CIFAR-10 Test Set', fontsize=14)
plt.tight_layout()
plt.show()

# TODO: Which classes are most often confused? Why?
# YOUR ANSWER HERE:

### Summary Questions

1. What is the receptive field of a 3×3 conv → 3×3 conv stack? How does it differ from a single 5×5 conv?
2. Why is MaxPooling used in CNNs? What is its effect on spatial resolution and translation invariance?
3. How does BatchNorm affect training stability and generalization?
4. Which CIFAR-10 classes are hardest to classify? Can you explain why?

**Your Answers:**

1. *TODO: Your answer here*

2. *TODO: Your answer here*

3. *TODO: Your answer here*

4. *TODO: Your answer here*